# Module 4 — RAG answer evaluation with LLM-as-a-judge (lessons 11–14)

Search metrics tell us whether the right document was *retrieved*. They don't tell us whether the final *answer*
is right. With no reference labels at scale, a second LLM grades each answer.

Data: `data/rag-answers.csv` from the course. For each ground-truth question, `answer_llm` is what the course's
RAG (gpt-5.4-mini) answered and `answer_orig` is the FAQ answer it should match. The judge here is a
**different model** (`openai/gpt-oss-120b` on Groq), which avoids a model grading its own output.

Free-tier budget (1,000 requests/day, 8k tokens/minute) → a random sample of 40 instead of all ~400 answers.

In [1]:
import random
from typing import Literal

import pandas as pd
from pydantic import BaseModel, Field

from eval_utils import llm_structured

answers = pd.read_csv("data/rag-answers.csv").to_dict(orient="records")
random.seed(42)
sample = random.sample(answers, 40)
len(answers), len(sample)

(395, 40)

## The judge (lesson 13)

Structured output (a Pydantic model) means each verdict parses into a typed object: no regex over free text.
`reasoning` comes *before* `score`, so the model writes its justification before committing to a label.

In [2]:
class AnswerEvaluation(BaseModel):
    reasoning: str = Field(description="Reasoning about the quality of the answer.")
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )


JUDGE_INSTRUCTIONS = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

JUDGE_PROMPT = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()


def judge(rec):
    result, usage = llm_structured(JUDGE_INSTRUCTIONS, JUDGE_PROMPT.format(**rec), AnswerEvaluation)
    return {"question": rec["question"], "score": result.score, "reasoning": result.reasoning,
            "prompt_tokens": usage.prompt_tokens, "completion_tokens": usage.completion_tokens}

## Judging the sample

Sequential on purpose: parallel calls (the lesson's `ThreadPoolExecutor`) only hit the per-minute limit sooner.

In [3]:
from tqdm.auto import tqdm

df_eval = pd.DataFrame([judge(rec) for rec in tqdm(sample)])
df_eval.score.value_counts()

  0%|          | 0/40 [00:00<?, ?it/s]

score
good    38
bad      2
Name: count, dtype: int64

In [4]:
for _, row in df_eval[df_eval.score == "bad"].iterrows():
    print("Q:", row.question)
    print("  ->", row.reasoning[:400])
    print()

Q: How do the free GPU hours work on these cloud options — is the limit monthly, weekly, or something else?
  -> The AI answer simply states "I don’t know," which provides no information about how free GPU hours work, quota cycles, or any of the details present in the original answer. It fails to convey the key points, so it is not semantically equivalent.

Q: What should I check if I want to use an open-source replacement for OpenAI in the intro module?
  -> The AI answer mentions checking the open-ai-alternatives.md file in the module 1 folder, which is part of the correct response, but it omits the reference to module 2 that the original answer includes. Since it misses a key piece of information, the answer is incomplete.



## Calibration: can the judge catch a clearly wrong answer?

A judge's "good" rate means nothing unless it also says "bad" when it should. Ten known-bad cases:
keep each question and its ground truth, but pair them with an AI answer to a **different** question.
A reliable judge should reject all ten.

In [5]:
random.seed(7)
pool = random.sample(answers, 20)
negatives = [
    {"question": a["question"], "answer_orig": a["answer_orig"], "answer_llm": b["answer_llm"]}
    for a, b in zip(pool[:10], pool[10:])
]
df_neg = pd.DataFrame([judge(rec) for rec in tqdm(negatives)])
print("known-bad answers marked bad:", (df_neg.score == "bad").sum(), "/", len(df_neg))

  0%|          | 0/10 [00:00<?, ?it/s]

known-bad answers marked bad: 10 / 10


In [6]:
print(f"tokens per verdict: prompt ~{df_eval.prompt_tokens.mean():.0f}, completion ~{df_eval.completion_tokens.mean():.0f}")
print(f"'good' rate on real answers: {(df_eval.score == 'good').mean():.0%}")

tokens per verdict: prompt ~518, completion ~271
'good' rate on real answers: 95%


**Findings**

- 38/40 (95%) of the course RAG answers are judged equivalent to the FAQ. Both "bad" verdicts hold up on reading:
  one answer is "I don't know", the other omits a pointer the FAQ gives.
- Calibration: the judge rejected **10/10** mismatched answers, so a "good" from it is not automatic.
- Limit of this check: mismatched answers are *easy* negatives (wrong topic). Subtle errors (a wrong number, a missing
  condition) are harder. Measuring those needs a small human-labelled set and agreement rate between human and judge.
- Cost of judging: ~520 prompt + ~270 completion tokens per verdict, a large share of it the model's hidden reasoning.
  Judging every production answer (module 5) roughly doubles LLM spend per request.

Contrast with my Sentinel AI project, whose judge marked 100% of answers NOT_RELEVANT for two of three prompt variants.
A judge that rejects nearly everything has the same problem as one that accepts everything. This calibration test
(feed it answers you *know* are good and bad) is what would catch that.